[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_13/MFM_ch13_triple_barrier/MFM_ch13_triple_barrier.ipynb)

# MFM_ch13_triple_barrier

**Modelling Financial Markets (MFM) — Chapter 13: Machine Learning in Finance**

Triple-barrier labelling of the S&P 500: volatility-scaled profit-taking and stop-loss barriers plus a vertical (time) barrier; illustration on 2022 events and comparison of label shares with fixed-horizon labels.

*Author: Daniel Traian Pele*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_13'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip -q install shap

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## Style and data

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'

SEED = 42


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


In [ ]:
import os
DATA_URL = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
SERIES = {'sp500': 'GSPC.INDX', 'vix': 'VIX.INDX', 'btc': 'BTC-USD.CC'}
START = {'sp500': '2000-01-01', 'vix': '2000-01-01', 'btc': '2014-09-17'}


def load_data(name='sp500', start=None, end='2026-09-18'):
    """OHLCV zilnic din data/market: copia locala, altfel fisierul din repo-ul GitHub MFM."""
    fname = SERIES[name] + '.csv'
    local = [os.path.join(d, 'data', 'market', fname) for d in ('.', '..', '../..', '../../..')]
    path = next((p for p in local if os.path.exists(p)), DATA_URL + fname)
    df = pd.read_csv(path, parse_dates=['date']).set_index('date')
    df = df.rename(columns=str.capitalize)[['Open', 'High', 'Low', 'Close', 'Volume']]
    return df.loc[start or START[name]:end]


spx = load_data('sp500')['Close']
vix = load_data('vix')['Close']
btc = load_data('btc')['Close']
log_spx = np.log(spx)
print('S&P 500:', spx.index[0].date(), '->', spx.index[-1].date(), len(spx), 'obs')
print('BTC    :', btc.index[0].date(), '->', btc.index[-1].date(), len(btc), 'obs')

## Helper functions

In [ ]:
def get_daily_vol(close, span=50):
    """Volatilitatea EWMA: radacina patrata a mediei EWMA a abaterilor patratice ale randamentelor log zilnice."""
    return np.log(close).diff().ewm(span=span).std()


def triple_barrier(close, events_idx, vol, pt=1.0, sl=1.0, horizon=10):
    """
    Pentru fiecare moment t0 din events_idx:
      bariera superioara = P_t0 * exp(+pt * vol_t0 * sqrt(horizon))
      bariera inferioara = P_t0 * exp(-sl * vol_t0 * sqrt(horizon))
      bariera verticala  = t0 + horizon zile de tranzactionare
    Evenimentele fara o fereastra completa de horizon zile (sfarsitul esantionului) sunt omise.
    Eticheta: +1 (atinge sus), -1 (atinge jos), sign(randament) la bariera verticala.
    Intoarce DataFrame cu t1 (momentul atingerii), ret, label, barrier.
    """
    logp = np.log(close)
    pos = {t: i for i, t in enumerate(close.index)}
    rows = []
    for t0 in events_idx:
        i0 = pos[t0]
        if i0 + horizon >= len(close) or np.isnan(vol.iloc[i0]):
            continue                                   # fereastra completa de h zile, altfel evenimentul e cenzurat
        i1 = i0 + horizon
        width = vol.iloc[i0] * np.sqrt(horizon)
        path = logp.iloc[i0 + 1:i1 + 1] - logp.iloc[i0]
        up = path[path >= pt * width].index.min() if pt > 0 else pd.NaT
        dn = path[path <= -sl * width].index.min() if sl > 0 else pd.NaT
        first = min([t for t in (up, dn) if pd.notna(t)], default=pd.NaT)
        if pd.isna(first):
            t1, barrier = close.index[i1], 'vertical'
            label = int(np.sign(path.iloc[-1])) if len(path) else 0
        else:
            t1 = first
            barrier = 'upper' if first == up else 'lower'
            label = 1 if barrier == 'upper' else -1
        rows.append((t0, t1, logp.loc[t1] - logp.iloc[i0], label, barrier, width))
    return pd.DataFrame(rows, columns=['t0', 't1', 'ret', 'label', 'barrier', 'width']).set_index('t0')

## Charts

In [ ]:
def fig_triple_barrier():
    close = spx.loc['2021-10-01':'2023-03-31']
    vol = get_daily_vol(spx).reindex(close.index)
    events = [pd.Timestamp(t) for t in ['2022-01-18', '2022-03-15', '2022-05-19', '2022-08-01', '2022-11-01']]
    events = [close.index[close.index.searchsorted(t)] for t in events]
    tb = triple_barrier(close, events, vol, pt=1.0, sl=1.0, horizon=15)

    fig, ax = plt.subplots(figsize=(7.2, 3.2))
    ax.plot(close.index, close.values, color=Purple, lw=0.8, label='S&P 500')
    colors = {'upper': Forest, 'lower': IDAred, 'vertical': Amber}
    for t0, row in tb.iterrows():
        p0 = close.loc[t0]
        i0 = close.index.get_loc(t0)
        tv = close.index[min(i0 + 15, len(close) - 1)]
        up, dn = p0 * np.exp(row['width']), p0 * np.exp(-row['width'])
        ax.add_patch(mpatches.Rectangle((t0, dn), tv - t0, up - dn, fill=True,
                                        fc=colors[row['barrier']], alpha=0.10, lw=0))
        ax.plot([t0, tv], [up, up], color=Forest, lw=0.9)
        ax.plot([t0, tv], [dn, dn], color=IDAred, lw=0.9)
        ax.plot([tv, tv], [dn, up], color=Amber, lw=0.9)
        ax.plot(t0, p0, 'o', color=MainBlue, ms=3.5)
        ax.plot(row['t1'], close.loc[row['t1']], marker='*', ms=9, color=colors[row['barrier']])
        ax.annotate(f"{'+1' if row['label'] > 0 else '-1'}", xy=(row['t1'], close.loc[row['t1']]),
                    xytext=(3, 6), textcoords='offset points', fontsize=8,
                    color=colors[row['barrier']], fontweight='bold')
    handles = [plt.Line2D([], [], color=Forest, label='Upper barrier (profit-taking)'),
               plt.Line2D([], [], color=IDAred, label='Lower barrier (stop-loss)'),
               plt.Line2D([], [], color=Amber, label='Vertical barrier (time limit)'),
               plt.Line2D([], [], color=MainBlue, marker='o', ls='', ms=4, label='Event $t_0$'),
               plt.Line2D([], [], color='black', marker='*', ls='', ms=8, label='First touch $t_1$')]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
    ax.set_ylabel('Index level')
    ax.set_title('Triple-barrier labelling: barriers at $\\pm\\sigma_{t_0}\\sqrt{h}$, $h=15$ days',
                 fontsize=9, loc='left')
    plt.tight_layout()
    save_fig('ch13_triple_barrier')

In [ ]:
def fig_label_comparison():
    vol = get_daily_vol(spx)
    idx = spx.index[250:-20]
    tb = triple_barrier(spx, idx, vol, pt=1.0, sl=1.0, horizon=10)
    fixed = np.sign(np.log(spx).shift(-10) - np.log(spx)).reindex(tb.index)
    shares = pd.DataFrame({
        'Fixed horizon (10d)': [np.mean(fixed > 0), np.mean(fixed < 0), 0.0],
        'Triple barrier': [np.mean(tb['barrier'] == 'upper'), np.mean(tb['barrier'] == 'lower'),
                           np.mean(tb['barrier'] == 'vertical')]},
        index=['+1 (up)', '-1 (down)', 'time-out'])
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8), gridspec_kw={'width_ratios': [1, 1.25]})
    x = np.arange(3)
    axes[0].bar(x - 0.18, shares.iloc[:, 0], 0.36, color=MainBlue, label=shares.columns[0])
    axes[0].bar(x + 0.18, shares.iloc[:, 1], 0.36, color=Amber, label=shares.columns[1])
    axes[0].set_xticks(x, shares.index)
    axes[0].set_ylabel('Share of labels')
    axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
    hold = (tb['t1'] - tb.index).dt.days
    axes[1].hist(hold[tb['barrier'] != 'vertical'], bins=np.arange(0, 17, 1), color=MainBlue, alpha=0.85)
    axes[1].set_xlabel('Calendar days until first touch')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Horizontal barrier hits', fontsize=8.5, loc='left')
    axes[0].set_title('S&P 500, 2001-2026', fontsize=8.5, loc='left')
    plt.tight_layout()
    save_fig('ch13_label_comparison')
    return shares

## Run

In [ ]:
fig_triple_barrier()
print(fig_label_comparison().round(3))

![ch13_triple_barrier](./ch13_triple_barrier.png)

![ch13_label_comparison](./ch13_label_comparison.png)

Output: `ch13_triple_barrier.pdf`, `ch13_label_comparison.pdf`